# Lab 6: build an MCP server

A tiny MCP server with two tools, a resource and a prompt, and two ways to talk to it. This
notebook follows [Lab 6 in the book](https://build-ai-handbook.vercel.app/lab-mcp.html): read each step there, then run
it here. The code is in `labs/06-mcp-server/`. The lab makes no model calls and costs nothing.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://build-ai-handbook.vercel.app/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

In [ ]:
# The MCP SDK is not in the shared requirements: install it only when it is missing.
try:
    import mcp  # noqa: F401
except ImportError:
    run_command("python", "-m", "pip", "install", "--quiet", "-r", "labs/06-mcp-server/requirements.txt")

## Step 1: keep the logic plain

`deposits.py` has no MCP in it, so call it directly. Ten thousand at 4.10% for one year is
10,410.00, and for two years 10,836.81. See [Step 1](https://build-ai-handbook.vercel.app/lab-mcp.html#logic).

In [ ]:
sys.path.insert(0, str(root / "labs" / "06-mcp-server"))
import deposits

print(deposits.RATES)
for years in (1, 2):
    print(deposits.compound_value("10000", "4.10", years))

A bad request raises the lab's own `InputError`, which the server will turn into a result
the model can read.

In [ ]:
try:
    deposits.compound_value("10000", "4.10", 99)
except deposits.InputError as err:
    print("InputError:", err)

## Step 2: what a server exposes

`server.py` builds two tools, a resource and a prompt with decorators. Print the decorators
and the function names to see the three kinds side by side. See
[Step 2](https://build-ai-handbook.vercel.app/lab-mcp.html#server).

In [ ]:
!grep -n -A1 "^@server\." labs/06-mcp-server/server.py

## Step 3: the messages on the wire

`wire.py` prints every raw JSON-RPC message, `>>` for the client and `<<` for the server,
for both protocol revisions. Look for the handshake in the first half and the `_meta` block
in the second. Each line is cut to 220 characters here. See
[Step 3](https://build-ai-handbook.vercel.app/lab-mcp.html#wire).

In [ ]:
def run_cut(script, width):
    """Run a lab script, print each line of its output cut to `width` characters, and stop if it fails."""
    done = subprocess.run([sys.executable, script], capture_output=True, text=True)
    for line in done.stdout.splitlines():
        print(line[:width] + (" ..." if len(line) > width else ""))
    if done.returncode:
        print(done.stderr)
        raise RuntimeError(f"{script} failed")


run_cut("labs/06-mcp-server/wire.py", 220)

## Step 4: use it through the SDK

`client.py` starts the server and uses every primitive. The last call breaks a rule on
purpose: look for `isError: True` and the reason beside it. See
[Step 4](https://build-ai-handbook.vercel.app/lab-mcp.html#client).

In [ ]:
run_cut("labs/06-mcp-server/client.py", 300)

## Step 5: the same server over HTTP

Start the server on a spare port, send it a request from Python, then stop it. The second
request carries a foreign `Origin` header and should get a 403. See
[Step 5](https://build-ai-handbook.vercel.app/lab-mcp.html#http).

In [ ]:
import json, urllib.error, urllib.request

PORT = 8799
server = subprocess.Popen(
    [sys.executable, "labs/06-mcp-server/server.py", "--http", "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
URL = f"http://127.0.0.1:{PORT}/mcp"
meta = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "0"}}
body = json.dumps({"jsonrpc": "2.0", "id": "d1", "method": "server/discover", "params": {"_meta": meta}}).encode()
headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream",
           "MCP-Protocol-Version": "2026-07-28", "Mcp-Method": "server/discover"}


def post(extra=None):
    request = urllib.request.Request(URL, data=body, headers={**headers, **(extra or {})}, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=5) as response:
            return response.status, response.read().decode()[:160]
    except urllib.error.HTTPError as err:
        return err.code, err.read().decode()[:160]


try:
    for _ in range(30):  # wait for the server to start listening
        try:
            first = post()
            break
        except OSError:
            time.sleep(0.5)
    print("plain request:  ", first)
    print("foreign Origin: ", post({"Origin": "https://attacker.example"}))
finally:
    server.terminate()
    server.wait(timeout=10)

## Step 6: connect it to an MCP host

A host is a separate application, so there is nothing to run here. Print the settings a
host would need, using full paths and the Python that has `mcp` installed. See
[Step 6](https://build-ai-handbook.vercel.app/lab-mcp.html#host).

In [ ]:
print(json.dumps({"mcpServers": {"handbook-rates": {
    "command": sys.executable,
    "args": [str(root / "labs" / "06-mcp-server" / "server.py")],
}}}, indent=2))

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 06-mcp-server

## Experiments

Try the experiments on the [lab page](https://build-ai-handbook.vercel.app/lab-mcp.html#experiments), such as adding a
tool of your own. To start, here is the sum for a deposit of your choice: change the numbers
and run it, then try a principal the server should refuse. Then tick the lab's checklist in
the book: your progress is kept in your browser, not in this notebook.

In [ ]:
print(deposits.compound_value("2500", deposits.RATES["bank-c"], 5))